In [1]:
import copy
import itertools
from collections import defaultdict
from pathlib import Path

import partitura as pt
import partitura.score

from src.utils.ansi_colors import Colors

In [2]:
# Parameters
PATH = Path("/home/gerel/Documents/datasets/nasap-dataset")
assert PATH.exists(), f"{PATH} does not exist."

# Analyse single score

In [3]:
# Investigate single score
score_path = PATH / "Beethoven/Piano_Sonatas/1-1/xml_score.musicxml"

score = pt.load_musicxml(score_path)
assert len(score.parts) == 1, f"Expected 1 part, found {len(score.parts)}."

part = score.parts[0]
print(part.pretty())

Part id="P1" name="Piano"
 │
 ├─ TimePoint t=0 quarter=30
 │   │
 │   └─ starting objects
 │       │
 │       ├─ 0-- Clef sign=G line=2 number=1
 │       ├─ 0-- Clef sign=F line=4 number=2
 │       ├─ 0--750 ConstantLoudnessDirection "p"
 │       ├─ 0--18240 ConstantTempoDirection "allegro" raw_text="Allegro"
 │       ├─ 0-- KeySignature fifths=-4, mode=None (Ab)
 │       ├─ 0--30 Measure number=1 name=1
 │       ├─ 0--30 Note id=n1 voice=1 staff=1 type=quarter articulations=(staccato) pitch=C4
 │       ├─ 0--3510 Page number=1
 │       ├─ 0--30 Rest id=n2 voice=5 staff=2 type=
 │       ├─ 0--630 System number=1
 │       └─ 0-- TimeSignature 2/2
 │
 ├─ TimePoint t=30 quarter=30
 │   │
 │   ├─ ending objects
 │   │   │
 │   │   ├─ 0--30 Measure number=1 name=1
 │   │   ├─ 0--30 Note id=n1 voice=1 staff=1 type=quarter articulations=(staccato) pitch=C4
 │   │   └─ 0--30 Rest id=n2 voice=5 staff=2 type=
 │   │
 │   └─ starting objects
 │       │
 │       ├─ 30--150 IncreasingLoudnessDirect

/home/gerel/Applications/partitura/partitura/directions.py:514: UserWarning: error parsing "con passione" (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))


In [44]:
class VoiceCompletionInfo:
    def __init__(self, notes: list[pt.score.GenericNote], quarter_note_duration: float= 30, error_msg: str | None = None):
        self._notes = notes
        self._quarter_note_duration = quarter_note_duration
        self.error_msg = error_msg

    @property
    def num_notes(self) -> int:
        return len(self._notes)

    @property
    def duration(self) -> float:
        return sum([n.duration for n in self._notes]) / self._quarter_note_duration


class MeasureCompletionInfo:
    def __init__(self, measure: pt.score.Measure, part: pt.score.Part):
        self._measure = measure
        self._part = part
        self._quarter_note_duration = part.quarter_duration_map(measure.start.t)

        self.ts = next(measure.start.iter_prev(cls=pt.score.TimeSignature, eq=True))
        self._voices = self._compute_voices()

    def _compute_voices(self):
        voices = defaultdict(list)
        for note in self._part.iter_all(
            cls=pt.score.GenericNote,
            start=self._measure.start,
            end=self._measure.end,
            include_subclasses=True,
        ):
            note: pt.score.GenericNote
            voices[note.voice].append(note)

        for voice, notes in voices.items():
            # Merge notes from same chord
            notes = itertools.groupby(notes, key=lambda n: n.start.t)
            notes = [next(g) for _, g in notes]
            if not all(n.start + n.duration == n_next.start for n, n_next in zip(notes[:-1], notes[1:])):
                voices[voice] = VoiceCompletionInfo(notes, self._quarter_note_duration, "Voice not contiguous.")
            else:
                voices[voice] = VoiceCompletionInfo(notes, self._quarter_note_duration)

        return voices

    @property
    def number(self) -> int:
        return self._measure.number

    @property
    def name(self) -> str:
        return self._measure.name

    @property
    def time_signature(self) -> tuple[int, int]:
        return self.ts.beats, self.ts.beat_type

    @property
    def nominal_duration(self) -> float:
        return self.ts.beats / self.ts.beat_type * 4

    @property
    def actual_duration(self) -> float:
        return self._measure.duration / self._part.quarter_duration_map(self._measure.start.t)

    @property
    def is_nominal(self) -> bool:
        return self.nominal_duration == self.actual_duration

    @property
    def voices(self) -> dict[int, VoiceCompletionInfo]:
        return self._voices

In [45]:
def analyse_measure_completion(part: pt.score.Part) -> list[MeasureCompletionInfo]:
    """
    Analyse the completion of measures in a part.

    :param part: Part to analyse.
    :return: List of MeasureCompletionInfo objects.
    """
    return [MeasureCompletionInfo(measure, part) for measure in part.iter_all(cls=pt.score.Measure)]

In [48]:
def analysis_summary(analysis: list[MeasureCompletionInfo]):
    for measure_info in analysis:
        dur_string = f"{measure_info.nominal_duration} beats"
        if measure_info.nominal_duration != measure_info.actual_duration:
            dur_string = f"{Colors.RED}{Colors.BOLD}{dur_string} [R={measure_info.actual_duration}]{Colors.END}"
        print(f"Measure {measure_info.number} ({measure_info.name}): {measure_info.time_signature[0]}/{measure_info.time_signature[1]} ({dur_string}).")
        for voice, voice_info in measure_info.voices.items():
            voice_string = f"  V{voice}: {voice_info.duration} beats ({voice_info.num_notes} notes)"
            if voice_info.error_msg is not None:
                voice_string = f"{Colors.RED}{Colors.BOLD}{voice_string} [{voice_info.error_msg}]{Colors.END}"
            if voice_info.duration < measure_info.actual_duration:
                voice_string = f"{Colors.GREEN}{Colors.BOLD}{voice_string}{Colors.END}"
            elif voice_info.duration > measure_info.actual_duration:
                voice_string = f"{Colors.RED}{Colors.BOLD}{voice_string}{Colors.END}"
            print(voice_string)
        # pprint(measure_info, sort_dicts=False)

    print("Results:")
    num_measure_diff = len([m for m in analysis if m.nominal_duration != m.actual_duration])
    print(f"  - {num_measure_diff} measures with different actual duration")
    if num_measure_diff > 0:
        measures_shorter = [m for m in analysis if m.nominal_duration > m.actual_duration]
        measures_longer = [m for m in analysis if m.nominal_duration < m.actual_duration]
        print(f"     - {len(measures_shorter)} measures shorter than expected")
        print(f"     - {len(measures_longer)} measures longer than expected")

    num_voices_diff = len([v for m in analysis for v in m.voices.values() if v.duration != m.actual_duration])
    print(f"  - {num_voices_diff} voices with different actual duration")
    if num_voices_diff > 0:
        voices_shorter = [v for m in analysis for v in m.voices.values() if v.duration < m.actual_duration]
        voices_longer = [v for m in analysis for v in m.voices.values() if v.duration > m.actual_duration]
        print(f"     - {len(voices_shorter)} voices shorter than expected")
        print(f"     - {len(voices_longer)} voices longer than expected")

In [50]:
analysis_summary(analyse_measure_completion(part))

Measure 1 (1): 12/8 (6.0 beats).
  V5: 6.0 beats (9 notes)
  V1: 6.0 beats (7 notes)
  V2: 6.0 beats (7 notes)
Measure 2 (2): 12/8 (6.0 beats).
  V1: 6.0 beats (11 notes)
  V2: 6.0 beats (12 notes)
  V5: 6.0 beats (12 notes)
Measure 3 (3): 12/8 (6.0 beats).
  V1: 6.0 beats (7 notes)
  V2: 6.0 beats (7 notes)
  V5: 6.0 beats (7 notes)
Measure 4 (4): 12/8 (6.0 beats).
  V5: 6.0 beats (14 notes)
  V1: 6.0 beats (1 notes)
Measure 5 (5): 12/8 (6.0 beats).
  V5: 6.0 beats (14 notes)
  V1: 6.0 beats (1 notes)
Measure 6 (6): 12/8 (6.0 beats).
  V5: 6.0 beats (14 notes)
  V1: 6.0 beats (9 notes)
Measure 7 (7): 12/8 (6.0 beats).
  V1: 6.0 beats (7 notes)
  V5: 6.0 beats (14 notes)
  V2: 6.0 beats (3 notes)
Measure 8 (8): 12/8 (6.0 beats).
  V1: 6.0 beats (10 notes)
  V5: 6.0 beats (14 notes)
Measure 9 (9): 12/8 (6.0 beats).
  V1: 6.0 beats (9 notes)
  V5: 6.0 beats (14 notes)
  V2: 6.0 beats (6 notes)
Measure 10 (10): 12/8 (6.0 beats).
  V1: 6.0 beats (10 notes)
  V5: 6.0 beats (14 notes)
Measur

In [51]:
part_filled = copy.deepcopy(part)

In [52]:
# Fill the part with rests
pt.score.fill_rests(part_filled, measurewise=True)
analysis_summary(analyse_measure_completion(part_filled))

Measure 1 (1): 12/8 (6.0 beats).
  V5: 6.0 beats (9 notes)
  V1: 6.0 beats (7 notes)
  V2: 6.0 beats (7 notes)
Measure 2 (2): 12/8 (6.0 beats).
  V1: 6.0 beats (11 notes)
  V2: 6.0 beats (12 notes)
  V5: 6.0 beats (12 notes)
Measure 3 (3): 12/8 (6.0 beats).
  V1: 6.0 beats (7 notes)
  V2: 6.0 beats (7 notes)
  V5: 6.0 beats (7 notes)
Measure 4 (4): 12/8 (6.0 beats).
  V5: 6.0 beats (14 notes)
  V1: 6.0 beats (1 notes)
Measure 5 (5): 12/8 (6.0 beats).
  V5: 6.0 beats (14 notes)
  V1: 6.0 beats (1 notes)
Measure 6 (6): 12/8 (6.0 beats).
  V5: 6.0 beats (14 notes)
  V1: 6.0 beats (9 notes)
Measure 7 (7): 12/8 (6.0 beats).
  V1: 6.0 beats (7 notes)
  V5: 6.0 beats (14 notes)
  V2: 6.0 beats (3 notes)
Measure 8 (8): 12/8 (6.0 beats).
  V1: 6.0 beats (10 notes)
  V5: 6.0 beats (14 notes)
Measure 9 (9): 12/8 (6.0 beats).
  V1: 6.0 beats (9 notes)
  V5: 6.0 beats (14 notes)
  V2: 6.0 beats (6 notes)
Measure 10 (10): 12/8 (6.0 beats).
  V1: 6.0 beats (10 notes)
  V5: 6.0 beats (14 notes)
Measur

# Analyse all scores

In [59]:
from tqdm import tqdm
import csv
import warnings

# Search for all scores
score_paths = list(PATH.rglob("xml_score.musicxml"))
print(f"Found {len(score_paths)} scores.")

# Result file
result_file = Path("/home/gerel/Documents/transcription/gramscore") / "metrical_errors.csv"
results = []
# result = {
#     "score_path": score_path,
#     "measure_number": measure_info.number,
#     "measure_name": measure_info.name,
#     "nominal_duration": measure_info.nominal_duration,
#     "actual_duration": measure_info.actual_duration,
#     "direction": "shorter" if measure_info.nominal_duration > measure_info.actual_duration else "longer",
# }

# %% Loop over them, stopping when finding a score with more/less beats than expected
for i, score_path in enumerate(pbar := tqdm(score_paths)):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            score = pt.load_musicxml(score_path)
    except Exception as e:
        print(f"Error loading {score_path}: {e}")
        continue
    assert len(score.parts) == 1, f"Expected 1 part, found {len(score.parts)}."
    part = score.parts[0]
    try:
        pt.score.fill_rests(part, measurewise=True)
    except Exception as e:
        print(f"Error filling rests in {score_path}: {e}")
        continue

    pbar.set_postfix_str(f"{score_path.relative_to(PATH)}")
    analysis = analyse_measure_completion(part)
    if not all([m.is_nominal for m in analysis]):
        for measure_info in analysis:
            if not measure_info.is_nominal:
                results.append({
                    "score_path": score_path,
                    "measure_number": measure_info.number,
                    "measure_name": measure_info.name,
                    "nominal_duration": measure_info.nominal_duration,
                    "actual_duration": measure_info.actual_duration,
                    "direction": "shorter" if measure_info.nominal_duration > measure_info.actual_duration else "longer",
                    "errors": " | ".join([f"V{voice}: {voice_info.error_msg}" for voice, voice_info in measure_info.voices.items() if voice_info.error_msg is not None]),
                })

# Save results
with open(result_file, "w") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

# Voix qui saute d'un staff à l'autre dans mesure 47 de Glinka/The_Lark/xml_score.musicxml

Found 239 scores.


  3%|▎         | 8/239 [00:03<01:39,  2.33it/s, Mozart/Piano_Sonatas/11-3/xml_score.musicxml]

Error loading /home/gerel/Documents/datasets/nasap-dataset/Balakirev/Islamey/xml_score.musicxml: Tuplet start time is after tuplet stop time


 16%|█▋        | 39/239 [00:20<01:03,  3.15it/s, Chopin/Etudes_op_25/10/xml_score.musicxml]       

Error filling rests in /home/gerel/Documents/datasets/nasap-dataset/Chopin/Etudes_op_25/2/xml_score.musicxml: list index out of range


 17%|█▋        | 41/239 [00:21<01:33,  2.12it/s, Chopin/Etudes_op_25/11/xml_score.musicxml]

... (37 outputs left out)

 97%|█████████▋| 233/239 [01:50<00:03,  1.97it/s, Schumann/Kreisleriana/4/xml_score.musicxml]                                    

Error loading /home/gerel/Documents/datasets/nasap-dataset/Schumann/Kreisleriana/1/xml_score.musicxml: Tuplet start time is after tuplet stop time


100%|██████████| 239/239 [01:51<00:00,  2.14it/s, Schumann/Kreisleriana/2/xml_score.musicxml]

Error loading /home/gerel/Documents/datasets/nasap-dataset/Schumann/Kreisleriana/1_no_first_repeat/xml_score.musicxml: Tuplet start time is after tuplet stop time
Error loading /home/gerel/Documents/datasets/nasap-dataset/Schumann/Kreisleriana/3/xml_score.musicxml: Tuplet start time is after tuplet stop time


In [61]:
pt.score.fill_rests(pt.load_musicxml("/home/gerel/Documents/datasets/asap-dataset/Chopin/Etudes_op_25/2/xml_score.musicxml").parts[0], measurewise=True)

IndexError: list index out of range

In [39]:
# score = pt.load_musicxml(PATH / "Glinka/The_Lark/xml_score.musicxml")  # m[46]
score = pt.load_musicxml(PATH / "Chopin/Barcarolle/xml_score.musicxml", ignore_invisible_objects=True)  # m[17]
part: pt.score.Part = score.parts[0]

# measure = part.measures[46]
measure = part.measures[17]
for note in part.iter_all(cls=pt.score.GenericNote, start=measure.start, end=measure.end, include_subclasses=True):
    if note.staff == 1:
        print(note)
pt.score.fill_rests(part, measurewise=True)
analysis = analyse_measure_completion(part)
analysis_summary(analysis)

48960--49440 Note id=n527 voice=1 staff=1 type=quarter pitch=D#4
48960--49440 Note id=n528 voice=1 staff=1 type=quarter pitch=B4
49440--49680 Rest id=n529 voice=1 staff=1 type=eighth
49680--49800 Note id=n530 voice=1 staff=1 type=16th pitch=E4
49680--49800 Note id=n531 voice=1 staff=1 type=16th pitch=C#5
49800--49920 Note id=n532 voice=1 staff=1 type=16th pitch=C#4
49800--49920 Note id=n533 voice=1 staff=1 type=16th pitch=A#4
49920--50040 Note id=n534 voice=1 staff=1 type=16th pitch=E4
49920--50040 Note id=n535 voice=1 staff=1 type=16th pitch=C#5
50040--50160 Note id=n536 voice=1 staff=1 type=16th pitch=B3
50040--50160 Note id=n537 voice=1 staff=1 type=16th pitch=G#4
50160--50280 Note id=n538 voice=1 staff=1 type=16th pitch=E4
50160--50280 Note id=n539 voice=1 staff=1 type=16th pitch=C#5
50280--50400 Note id=n540 voice=1 staff=1 type=16th pitch=F#3
50280--50400 Note id=n541 voice=1 staff=1 type=16th pitch=F#4
50400--50880 Note id=n542 voice=1 staff=1 type=quarter pitch=D#4
50400--50880

AssertionError: Voice is not contiguous.

In [11]:
[str(note) for note in analysis[46]._voices[2]._notes]

NameError: name 'analysis' is not defined

In [43]:
# score = pt.load_musicxml(PATH / "Glinka/The_Lark/xml_score.musicxml")
score = pt.load_musicxml(PATH / "Chopin/Barcarolle/xml_score.musicxml", ignore_invisible_objects=True)
part: pt.score.Part = score.parts[0]
pt.score.fill_rests(part, measurewise=True)

# measure47: pt.score.Measure = part.measures[46]
measure47: pt.score.Measure = part.measures[16]
print(f"{measure47.number = }")
print(f"{measure47.name = }")
voices = defaultdict(list)

voice_format = {
    1: Colors.BLUE,
    2: Colors.GREEN,
    5: Colors.RED,
    6: Colors.PURPLE,
}
staff_format = {
    1: Colors.BOLD,
    2: Colors.ITALIC + "  ",
}
for voice, format in voice_format.items():
    print(f"{format}Voice {voice}{Colors.END}")
print()
for staff, format in staff_format.items():
    print(f"{format}Staff {staff}{Colors.END}")
print()

for note in part.iter_all(cls=pt.score.GenericNote, start=measure47.start, end=measure47.end, include_subclasses=True):
    note: pt.score.GenericNote
    voices[note.voice].append(note)
    print(f"{voice_format[note.voice]}{staff_format[note.staff]}{note}{Colors.END}")

measure47.number = 17
measure47.name = '17'
Voice 1
Voice 2
Voice 5
Voice 6

Staff 1
  Staff 2

46080--46800 Note id=n485 voice=1 staff=1 type=quarter. articulations=(accent) pitch=E4
46080--46800 Note id=n486 voice=1 staff=1 type=quarter. pitch=G#4
46080--46800 Note id=n487 voice=1 staff=1 type=quarter. pitch=C#5
  46080--46320 Note id=n515 voice=5 staff=2 type=eighth tie_group=n483+n515 pitch=C#2
46080--46080 GraceNote id=n484 voice=1 staff=1 type=32nd pitch=F#4 main_note=46080--46800 Note id=n485 voice=1 staff=1 type=quarter. articulations=(accent) pitch=E4
46080--48120 Rest id=None voice=2 staff=1 type=whole_1/2
  46320--46560 Note id=n516 voice=5 staff=2 type=eighth pitch=G#2
  46560--46800 Note id=n517 voice=5 staff=2 type=eighth tie_group=n517+n518 pitch=G#3
46800--47160 Note id=n488 voice=1 staff=1 type=eighth. pitch=E4
46800--47160 Note id=n489 voice=1 staff=1 type=eighth. pitch=G#4
46800--47160 Note id=n490 voice=1 staff=1 type=eighth. pitch=C#5
  46800--47040 Note id=n518 vo